# 안전부담점수 데이터 검증 노트북

원본 분석에서 산출된 안전부담점수 CSV를 기준으로 구성요소 상관, VIF, 클러스터링 품질, 재현성을 독립적으로 점검합니다. 모든 표시 점수는 소수점 2자리로 반올림합니다.


## 0. 데이터 로드와 안전부담점수 구성요소 재생성

이 단계에서는 원본 분석에서 저장된 연도별 안전부담점수 CSV와 3년 평균 CSV를 불러옵니다. 검증 노트북은 원본 노트북의 중간 변수에 의존하지 않고, 저장된 산출물을 기준으로 같은 구성요소를 다시 만들어 점검합니다.

확인하는 내용은 다음과 같습니다.

- `발생_1만명당`, `검거율`, `총출동지표_12개월환산_1만명당`이 정상적으로 불러와졌는지 확인합니다.
- `미검거율 = 1 - 검거율`을 만든 뒤, 연도별 min-max 정규화로 세 구성요소를 재생성합니다.
- 기존 `안전부담점수`와 재계산한 점수가 같은지 이후 재현성 검증에서 확인할 수 있도록 준비합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy.stats import kruskal, mannwhitneyu
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

def set_korean_font():
    from matplotlib import font_manager, rcParams

    font_candidates = [
        'Malgun Gothic',
        '맑은 고딕',
        'AppleGothic',
        'NanumGothic',
        'Noto Sans CJK KR',
        'Noto Sans KR',
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in font_candidates:
        if font_name in available_fonts:
            rcParams['font.family'] = font_name
            rcParams['axes.unicode_minus'] = False
            print(f'한글 폰트 설정: {font_name}')
            return font_name

    rcParams['axes.unicode_minus'] = False
    print('사용 가능한 한글 폰트를 찾지 못했습니다. 그래프 한글이 깨지면 Malgun Gothic 또는 NanumGothic을 설치/등록하세요.')
    return None

set_korean_font()


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'branch_SG' / 'safety_analystic' / 'outputs' / 'gu_safety_burden_score_25_by_year.csv').exists():
            return candidate
        if (candidate / 'outputs' / 'gu_safety_burden_score_25_by_year.csv').exists():
            return candidate.parents[1]
    raise FileNotFoundError(
        'gu_safety_burden_score_25_by_year.csv를 찾을 수 없습니다. '
        '노트북을 저장소 내부에서 실행하고 있는지 확인하세요.'
    )

BASE_DIR = find_project_root()
SAFETY_DIR = BASE_DIR / 'branch_SG' / 'safety_analystic'
OUTPUT_DIR = SAFETY_DIR / 'outputs'
BY_YEAR_PATH = OUTPUT_DIR / 'gu_safety_burden_score_25_by_year.csv'
AVG3_PATH = OUTPUT_DIR / 'gu_safety_burden_score_25_3year_avg.csv'

print(f'프로젝트 루트: {BASE_DIR}')
print(f'연도별 점수 파일: {BY_YEAR_PATH}')
print(f'3년 평균 점수 파일: {AVG3_PATH}')

def minmax(s):
    s = pd.to_numeric(s, errors='coerce')
    rng = s.max() - s.min()
    if pd.isna(rng) or rng == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / rng

def fdr_bh(p_values):
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    return out

by_year = pd.read_csv(BY_YEAR_PATH)
avg3 = pd.read_csv(AVG3_PATH)

by_year['연도'] = by_year['연도'].astype(int)
by_year['미검거율'] = 1 - by_year['검거율']
by_year['발생_1만명당_norm'] = by_year.groupby('연도')['발생_1만명당'].transform(minmax)
by_year['미검거율_norm'] = by_year.groupby('연도')['미검거율'].transform(minmax)
by_year['출동_1만명당_norm'] = by_year.groupby('연도')['총출동지표_12개월환산_1만명당'].transform(minmax)
by_year['안전부담점수_재계산'] = (
    by_year['발생_1만명당_norm'] * 0.45
    + by_year['미검거율_norm'] * 0.25
    + by_year['출동_1만명당_norm'] * 0.30
) * 100
by_year['안전부담점수_25점_재계산'] = by_year['안전부담점수_재계산'] / 4
avg3['미검거율_3년평균'] = 1 - avg3['검거율_3년평균']

display(by_year.head().round(2))
display(avg3.head().round(2))


## 1. 구성요소와 안전부담점수 상관계수

이 단계에서는 안전부담점수의 세 구성요소와 최종 점수 사이의 관계를 먼저 확인합니다. 점수는 가중합으로 만들어졌기 때문에 최종 점수와 구성요소 간 상관이 어느 정도 나타나는 것은 자연스럽지만, 특정 구성요소 하나에 과도하게 끌리는지도 함께 봐야 합니다.

확인하는 내용은 다음과 같습니다.

- 구성요소끼리 이미 너무 비슷하게 움직이는지 확인합니다.
- `안전부담점수`가 어떤 구성요소와 가장 강하게 연결되는지 확인합니다.
- Pearson은 선형 관계, Spearman은 순위 관계를 보기 위해 함께 계산합니다.

해석 기준은 절대값 기준으로 대략 `0.30 미만`은 약한 관계, `0.30~0.50`은 중간 관계, `0.50 이상`은 비교적 강한 관계로 봅니다.


In [ ]:
safety_score_validation_cols = [
    '발생_1만명당_norm',
    '미검거율_norm',
    '출동_1만명당_norm',
    '안전부담점수',
]

safety_score_validation = by_year[safety_score_validation_cols].dropna().copy()

pearson_corr = safety_score_validation.corr(method='pearson').round(2)
spearman_corr = safety_score_validation.corr(method='spearman').round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    pearson_corr,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[0],
    cbar_kws={'label': 'Pearson r'},
)
axes[0].set_title('안전부담점수 구성요소 Pearson 상관')
sns.heatmap(
    spearman_corr,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[1],
    cbar_kws={'label': 'Spearman rho'},
)
axes[1].set_title('안전부담점수 구성요소 Spearman 상관')
plt.tight_layout()
plt.show()

display(pearson_corr)
display(spearman_corr)


## 2. VIF 점검

이 단계에서는 안전부담점수 구성요소끼리 서로 과도하게 겹치는지 확인합니다. VIF는 한 변수가 나머지 변수들로 얼마나 잘 설명되는지를 보는 지표입니다.

이 코드를 실행하는 이유는 다음과 같습니다.

- 세 구성요소가 서로 독립적인 정보를 충분히 담고 있는지 확인합니다.
- 특정 지표들이 비슷한 현상을 중복 반영해 안전부담점수가 한쪽으로 치우칠 가능성을 점검합니다.
- VIF가 높다면 이후 해석에서 “구성요소 중복 가능성”을 명시할 근거가 됩니다.

해석 기준은 일반적으로 `VIF < 5`는 양호, `5 이상`은 주의, `10 이상`은 강한 다중공선성 가능성으로 봅니다.


In [ ]:
vif_features = ['발생_1만명당_norm', '미검거율_norm', '출동_1만명당_norm']
vif_data = by_year[vif_features].dropna().copy()

vif_rows = []
for target_col in vif_features:
    other_cols = [col for col in vif_features if col != target_col]
    model = LinearRegression()
    model.fit(vif_data[other_cols], vif_data[target_col])
    r2 = model.score(vif_data[other_cols], vif_data[target_col])
    vif = np.inf if np.isclose(1 - r2, 0) else 1 / (1 - r2)
    vif_rows.append({
        '변수': target_col,
        'R2': r2,
        'VIF': vif,
        '판정': '주의' if vif >= 5 else '양호',
    })

vif_result = pd.DataFrame(vif_rows)
vif_result[['R2', 'VIF']] = vif_result[['R2', 'VIF']].round(2)
display(vif_result)


## 3. 자치구 클러스터링 재현

이 단계에서는 원본 노트북의 `분석 아이디어 8. 자치구 클러스터링`을 검증 노트북 안에서 다시 재현합니다. 클러스터링 검증을 하려면 먼저 동일한 입력 변수와 동일한 방식으로 군집을 다시 만들어야 합니다.

사용하는 입력 변수는 다음과 같습니다.

- `발생_1만명당_3년평균`
- `검거율_3년평균`
- `총출동지표_12개월환산_1만명당_3년평균`

세 변수는 스케일이 다르기 때문에 `StandardScaler`로 표준화한 뒤 KMeans를 적용합니다. 현재 원본 분석 흐름에 맞춰 기본 클러스터 수는 `k=4`로 둡니다.


In [ ]:
cluster_features = [
    '발생_1만명당_3년평균',
    '검거율_3년평균',
    '총출동지표_12개월환산_1만명당_3년평균',
]

cluster_df = avg3.dropna(subset=cluster_features).copy()
X = StandardScaler().fit_transform(cluster_df[cluster_features])
k = min(4, len(cluster_df) - 1)

cluster_df['cluster'] = KMeans(n_clusters=k, random_state=42, n_init=50).fit_predict(X)
cluster_profile = cluster_df.groupby('cluster', as_index=False).agg(
    자치구수=('자치구', 'count'),
    발생_1만명당_3년평균=('발생_1만명당_3년평균', 'mean'),
    검거율_3년평균=('검거율_3년평균', 'mean'),
    총출동지표_12개월환산_1만명당_3년평균=('총출동지표_12개월환산_1만명당_3년평균', 'mean'),
    안전부담점수_3년평균=('안전부담점수_3년평균', 'mean'),
)
cluster_profile = cluster_profile.sort_values('안전부담점수_3년평균', ascending=False).reset_index(drop=True)
cluster_profile['cluster_name'] = [
    f'{i + 1}. 안전부담 {label}'
    for i, label in enumerate(['높음', '중상', '중하', '낮음'][:len(cluster_profile)])
]
cluster_name_map = dict(zip(cluster_profile['cluster'], cluster_profile['cluster_name']))
cluster_df['cluster_name'] = cluster_df['cluster'].map(cluster_name_map)

cluster_profile_display = cluster_profile[
    [
        'cluster',
        'cluster_name',
        '자치구수',
        '발생_1만명당_3년평균',
        '검거율_3년평균',
        '총출동지표_12개월환산_1만명당_3년평균',
        '안전부담점수_3년평균',
    ]
].round(2)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=cluster_df,
    x='발생_1만명당_3년평균',
    y='총출동지표_12개월환산_1만명당_3년평균',
    hue='cluster_name',
    size='검거율_3년평균',
    palette='tab10',
    sizes=(60, 220),
)
plt.title('자치구 주민 기준 안전 지표 클러스터')
plt.xlabel('발생_1만명당_3년평균')
plt.ylabel('총출동지표_12개월환산_1만명당_3년평균')
plt.legend(title='클러스터 유형', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()

display(cluster_profile_display)
display(cluster_df.sort_values(['cluster_name', '안전부담점수_3년평균']).round(2))


## 4. 클러스터링 검증

이 단계에서는 현재 사용한 클러스터 수가 데이터 구조에 어느 정도 맞는지 확인합니다. KMeans는 사용자가 클러스터 수를 정해야 하므로, 여러 후보 `k`를 비교해 현재 선택을 점검하는 과정이 필요합니다.

확인하는 지표는 다음과 같습니다.

- `Silhouette`: 높을수록 군집 내부는 가깝고 군집 간은 멀다는 뜻입니다.
- `Calinski_Harabasz`: 높을수록 군집 분리가 비교적 좋다는 뜻입니다.
- `Davies_Bouldin`: 낮을수록 군집 간 구분이 좋다는 뜻입니다.

이 검증은 “정답 클러스터 수”를 확정하기보다는, 현재 `k=4`가 무리한 선택인지 아닌지를 확인하는 참고 근거로 사용합니다.


In [ ]:
cluster_validation_rows = []
max_k = min(8, len(cluster_df) - 1)
for candidate_k in range(2, max_k + 1):
    labels = KMeans(n_clusters=candidate_k, random_state=42, n_init=50).fit_predict(X)
    cluster_validation_rows.append({
        '클러스터수': candidate_k,
        'Silhouette': silhouette_score(X, labels),
        'Calinski_Harabasz': calinski_harabasz_score(X, labels),
        'Davies_Bouldin': davies_bouldin_score(X, labels),
        '현재선택': candidate_k == k,
    })

cluster_validation = pd.DataFrame(cluster_validation_rows)
metric_cols = ['Silhouette', 'Calinski_Harabasz', 'Davies_Bouldin']
cluster_validation[metric_cols] = cluster_validation[metric_cols].round(2)

current_cluster_size = (
    cluster_df.groupby('cluster_name', as_index=False)
    .agg(
        자치구수=('자치구', 'count'),
        평균_안전부담점수_3년평균=('안전부담점수_3년평균', 'mean'),
    )
    .sort_values('평균_안전부담점수_3년평균', ascending=False)
)
current_cluster_size['평균_안전부담점수_3년평균'] = current_cluster_size['평균_안전부담점수_3년평균'].round(2)

display(cluster_validation)
display(current_cluster_size)


## 5. 재현성 검증

이 단계에서는 같은 입력 데이터로 같은 결과가 다시 나오는지 확인합니다. 안전부담점수 산식은 랜덤 요소가 없으므로 재계산 결과가 기존 점수와 같아야 합니다. 반면 KMeans는 초기값에 영향을 받을 수 있어 seed 변화에 따른 안정성을 봅니다.

확인하는 내용은 다음과 같습니다.

- 기존 안전부담점수와 재계산 점수의 최대 차이가 `0.00`인지 확인합니다.
- 여러 seed로 KMeans를 반복했을 때 기준 seed 42와 군집 배정이 얼마나 비슷한지 확인합니다.
- `ARI_vs_seed_42`는 1에 가까울수록 기준 seed와 거의 같은 군집 결과라는 뜻입니다.


In [ ]:
score_repro_check = pd.DataFrame({
    '검증항목': ['안전부담점수 재계산 최대차이', '안전부담점수_25점 재계산 최대차이'],
    '값': [
        (by_year['안전부담점수'] - by_year['안전부담점수_재계산']).abs().max(),
        (by_year['안전부담점수_25점'] - by_year['안전부담점수_25점_재계산']).abs().max(),
    ],
})
score_repro_check['값'] = score_repro_check['값'].round(2)

base_labels = KMeans(n_clusters=k, random_state=42, n_init=50).fit_predict(X)
seed_rows = []
for seed in [0, 1, 7, 42, 100, 2024, 2025]:
    labels = KMeans(n_clusters=k, random_state=seed, n_init=50).fit_predict(X)
    seed_rows.append({
        'seed': seed,
        'ARI_vs_seed_42': adjusted_rand_score(base_labels, labels),
        'Silhouette': silhouette_score(X, labels),
        'Davies_Bouldin': davies_bouldin_score(X, labels),
    })

cluster_repro_check = pd.DataFrame(seed_rows)
cluster_repro_check[['ARI_vs_seed_42', 'Silhouette', 'Davies_Bouldin']] = (
    cluster_repro_check[['ARI_vs_seed_42', 'Silhouette', 'Davies_Bouldin']].round(2)
)

display(score_repro_check)
display(cluster_repro_check)
